# Part 2: Unguided Exercise - Customer Churn Prediction
Build a churn prediction model

## Step 1: Load and Exploring the Data

In [116]:
import pandas as pd

raw_churn_data = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Print the shape of the dataset (number of rows and columns)
print(raw_churn_data.shape)

# The dataset is a Bunch object with 'data', 'target', 'feature_names', etc.
print(f"\nDataset type: {type(raw_churn_data)}")
print(f"Number of samples: {len(raw_churn_data)}")
print(f"Number of features: {raw_churn_data.shape[1] - 1}")  # Subtract 1 for the target column
print(f"Target classes: {raw_churn_data['Churn'].unique()}")

# To show the the table preview
# raw_churn_data.head()

(7043, 21)

Dataset type: <class 'pandas.core.frame.DataFrame'>
Number of samples: 7043
Number of features: 20
Target classes: ['No' 'Yes']


## Step 2: Data Preprocessing

### 2.1. Check missing data

In [117]:
# Basic statistics
print("\n" + "="*50)
print("BASIC STATISTICS")
print("="*50)
print(raw_churn_data.describe())
 
# Check for missing values
print("\n" + "="*50)
print("MISSING VALUES")
print("="*50)
missing = raw_churn_data.isnull().sum()
if missing.sum() == 0:
    print("✓ No missing values found!")
else:
    print(missing[missing > 0])



BASIC STATISTICS
       SeniorCitizen       tenure  MonthlyCharges
count    7043.000000  7043.000000     7043.000000
mean        0.162147    32.371149       64.761692
std         0.368612    24.559481       30.090047
min         0.000000     0.000000       18.250000
25%         0.000000     9.000000       35.500000
50%         0.000000    29.000000       70.350000
75%         0.000000    55.000000       89.850000
max         1.000000    72.000000      118.750000

MISSING VALUES
✓ No missing values found!


There's no missing values!

### 2.2 Convert categorical variables to numeric

In [118]:
# Let's use df for consistency with the rest of the code,
# and keep the raw data in case we need to refer back to it later
df = raw_churn_data.copy()

# Looking at one column at a time..

# gender (Female/Male, so it needs its own map and can't go in the Yes/No loop below)
df['gender'] = raw_churn_data['gender'].map({'Female': 0, 'Male': 1})
print("gender")
print(f"{list(raw_churn_data['gender'].unique())}")
print(f"0: {(df['gender'] == 0).sum()}, 1: {(df['gender'] == 1).sum()}, NaN: {df['gender'].isna().sum()}")


print("\nyes/no columns")
# The remaining Yes/No columns, all converted the same way
yes_no_columns = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for column_name in yes_no_columns:
    df[column_name] = raw_churn_data[column_name].map({'No': 0, 'Yes': 1})
    print(f"{column_name}: {list(raw_churn_data[column_name].unique())} -> 0: {(df[column_name] == 0).sum()}, 1: {(df[column_name] == 1).sum()}, NaN: {df[column_name].isna().sum()}")

print("\nContract mapping")
# Contract has a natural order, so it gets an ordinal map
df['Contract'] = raw_churn_data['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})
print(f"Contract: {list(raw_churn_data['Contract'].unique())} -> 0: {(df['Contract'] == 0).sum()}, 1: {(df['Contract'] == 1).sum()}, 2: {(df['Contract'] == 2).sum()}, NaN: {df['Contract'].isna().sum()}")

print("\nTotals to num")
# TotalCharges is stored as text, so it needs converting to a number (blank values become 0)
# New customers (tenure 0) have no charges yet, so their blank TotalCharges is really 0
df['TotalCharges'] = pd.to_numeric(raw_churn_data['TotalCharges'], errors='coerce').fillna(0)
print(f"TotalCharges: dtype {df['TotalCharges'].dtype}, NaN: {df['TotalCharges'].isna().sum()}")

print("\none_hot_columns")

# The remaining text columns have no natural order, so they get one-hot encoded
one_hot_columns = [
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod',
]
for column_name in one_hot_columns:
    print(f"{column_name}: {list(raw_churn_data[column_name].unique())}")
df = pd.get_dummies(df, columns=one_hot_columns, drop_first=True, dtype=int)
print(f"Columns after encoding: {df.shape[1]}")

gender
['Female', 'Male']
0: 3488, 1: 3555, NaN: 0

yes/no columns
Partner: ['Yes', 'No'] -> 0: 3641, 1: 3402, NaN: 0
Dependents: ['No', 'Yes'] -> 0: 4933, 1: 2110, NaN: 0
PhoneService: ['No', 'Yes'] -> 0: 682, 1: 6361, NaN: 0
PaperlessBilling: ['Yes', 'No'] -> 0: 2872, 1: 4171, NaN: 0
Churn: ['No', 'Yes'] -> 0: 5174, 1: 1869, NaN: 0

Contract mapping
Contract: ['Month-to-month', 'One year', 'Two year'] -> 0: 3875, 1: 1473, 2: 1695, NaN: 0

Totals to num
TotalCharges: dtype float64, NaN: 0

one_hot_columns
MultipleLines: ['No phone service', 'No', 'Yes']
InternetService: ['DSL', 'Fiber optic', 'No']
OnlineSecurity: ['No', 'Yes', 'No internet service']
OnlineBackup: ['Yes', 'No', 'No internet service']
DeviceProtection: ['No', 'Yes', 'No internet service']
TechSupport: ['No', 'Yes', 'No internet service']
StreamingTV: ['No', 'Yes', 'No internet service']
StreamingMovies: ['No', 'Yes', 'No internet service']
PaymentMethod: ['Electronic check', 'Mailed check', 'Bank transfer (automatic)',

In [119]:
# customerID is a unique label per customer, not something the model can learn from
model_data = df.drop(columns='customerID')
print(f"Columns before: {df.shape[1]}, after dropping customerID: {model_data.shape[1]}")

Columns before: 31, after dropping customerID: 30


In [120]:
# Separate features and target
X = model_data.drop(columns='Churn')  # All columns except 'Churn'
y = model_data['Churn']  # Target column (0 = stayed, 1 = churned)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (7043, 29)
Target shape: (7043,)


## Step 3: Split the Data

In [121]:
from sklearn.model_selection import train_test_split

print("\n" + "="*50)
print("DATA SPLIT")
print("="*50)

# 80% training, 20% testing
# stratify=y keeps the stayed/churned ratio the same in both sets
# random_state=42 makes the split reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Training set: {X_train.shape[0]} customers, {X_train.shape[1]} features")
print(f"Test set:     {X_test.shape[0]} customers, {X_test.shape[1]} features")

print(f"\nChurn rate overall:  {y.mean()*100:.1f}%")
print(f"Churn rate in train: {y_train.mean()*100:.1f}%")
print(f"Churn rate in test:  {y_test.mean()*100:.1f}%")


DATA SPLIT
Training set: 5634 customers, 29 features
Test set:     1409 customers, 29 features

Churn rate overall:  26.5%
Churn rate in train: 26.5%
Churn rate in test:  26.5%


## Step 4: Train a KNN Model

In [122]:
from sklearn.neighbors import KNeighborsClassifier

# n_neighbors=5 - looks at the 5 closest customers to make a prediction
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

print("KNN classifier trained successfully!")
print(f"Number of neighbors (k): {knn.n_neighbors}")

KNN classifier trained successfully!
Number of neighbors (k): 5


# 5 Making a Prediction

## 5.1 Make predictions on the test set

In [123]:
y_train_pred = knn.predict(X_train)
y_test_pred = knn.predict(X_test)

# How many predictions were done successfully it total:
# One prediction per customer: 0 = predicted to stay, 1 = predicted to churn
print(f"\nTotal predictions intraining set: {len(y_train_pred)}")
print(f"Total predictions in test set: {len(y_test_pred)}")


Total predictions intraining set: 5634
Total predictions in test set: 1409


## 5.2 Calculating accuracy, precision, and recall 


In [124]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall = recall_score(y_train, y_train_pred)

test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred)
test_recall = recall_score(y_test, y_test_pred)

print(f"Train accuracy:  {train_accuracy:.3f}")
print(f"Train precision: {train_precision:.3f}")
print(f"Train recall:    {train_recall:.3f}")
print("-")
print(f"Test accuracy:  {test_accuracy:.3f}")
print(f"Test precision: {test_precision:.3f}")
print(f"Test recall:    {test_recall:.3f}")

Train accuracy:  0.828
Train precision: 0.726
Train recall:    0.567
-
Test accuracy:  0.766
Test precision: 0.579
Test recall:    0.430


## 5.2 Confusion Matrix

In [127]:
from sklearn.metrics import confusion_matrix

# Confusion matrix as a readable table
# NOTE: encoding is 0 = stayed, 1 = churned
confusion = confusion_matrix(y_test, y_test_pred)
stayed, churned = 'stayed', 'churned'

rows = [
    [f"Actual {stayed} ({confusion[0].sum()})",  f"{confusion[0,0]}  correct",         f"{confusion[0,1]}  false alarm"],
    [f"Actual {churned} ({confusion[1].sum()})", f"{confusion[1,0]}  MISSED CHURNER",  f"{confusion[1,1]}  correct"],
]
header = ["", f"Predicted {stayed}", f"Predicted {churned}"]

column_widths = [max(len(row[i]) for row in [header] + rows) + 2 for i in range(3)]
border = lambda left, middle, right: left + middle.join("─" * width for width in column_widths) + right
format_row = lambda cells: "│" + "│".join(f" {cell:<{width - 1}}" for cell, width in zip(cells, column_widths)) + "│"

print("=== Confusion Matrix (test set) ===")
print(border("┌", "┬", "┐"))
print(format_row(header))
print(border("├", "┼", "┤"))
print(format_row(rows[0]))
print(border("├", "┼", "┤"))
print(format_row(rows[1]))
print(border("└", "┴", "┘"))

=== Confusion Matrix (test set) ===
┌──────────────────────┬─────────────────────┬───────────────────┐
│                      │ Predicted stayed    │ Predicted churned │
├──────────────────────┼─────────────────────┼───────────────────┤
│ Actual stayed (1035) │ 918  correct        │ 117  false alarm  │
├──────────────────────┼─────────────────────┼───────────────────┤
│ Actual churned (374) │ 213  MISSED CHURNER │ 161  correct      │
└──────────────────────┴─────────────────────┴───────────────────┘


## 5.3 Classification report

In [128]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_test_pred, target_names=['stayed', 'churned']))

              precision    recall  f1-score   support

      stayed       0.81      0.89      0.85      1035
     churned       0.58      0.43      0.49       374

    accuracy                           0.77      1409
   macro avg       0.70      0.66      0.67      1409
weighted avg       0.75      0.77      0.75      1409



## 5.4 Interpretraion: 

1. Accuracy: About 73.5% of customers don't churn, so a model that always predicts "stays" would score about 73.5%. The test accuracy of 76.6% beats that by only about 3 points.

2. Recall: The model catches only about 43% of the customers who actually churn, so it misses about 57% of them. 

3. Precision: When it flags someone as a churner, it's right about 58% of the time. So about 4 in 10 flags are false alarms.


# 6 Improving the model

## Checking K number

In [131]:
# Experiment with different K values
print("EXPERIMENTING WITH DIFFERENT K VALUES")

k_values = [1, 10, 100]
results = []
stayed, churned = 'stayed', 'churned'  # encoding: 0 = stayed, 1 = churned

for k in k_values:
    # Create and train model
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train, y_train)

    # Make predictions
    y_pred_temp = knn_temp.predict(X_test)

    # Calculate accuracy, precision and recall
    accuracy = accuracy_score(y_test, y_pred_temp)
    precision = precision_score(y_test, y_pred_temp)
    recall = recall_score(y_test, y_pred_temp)

    results.append({
        'K': k,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall
    })

    print(f"\nK={k:2d}: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}")

    # Confusion matrix as a readable table (uses this K's predictions)
    # Helps me understand what the model is doing with different K values, what is the real world impact of Accuracy, Precision, Recall changes
    confusion = confusion_matrix(y_test, y_pred_temp)

    rows = [
        [f"Actual {stayed} ({confusion[0].sum()})",  f"{confusion[0,0]}  correct",         f"{confusion[0,1]}  false alarm"],
        [f"Actual {churned} ({confusion[1].sum()})", f"{confusion[1,0]}  MISSED CHURNER",  f"{confusion[1,1]}  correct"],
    ]
    header = ["", f"Predicted {stayed}", f"Predicted {churned}"]

    column_widths = [max(len(row[i]) for row in [header] + rows) + 2 for i in range(3)]
    border = lambda left, middle, right: left + middle.join("─" * width for width in column_widths) + right
    format_row = lambda cells: "│" + "│".join(f" {cell:<{width - 1}}" for cell, width in zip(cells, column_widths)) + "│"

    print(border("┌", "┬", "┐"))
    print(format_row(header))
    print(border("├", "┼", "┤"))
    print(format_row(rows[0]))
    print(border("├", "┼", "┤"))
    print(format_row(rows[1]))
    print(border("└", "┴", "┘"))

EXPERIMENTING WITH DIFFERENT K VALUES

K= 1: Accuracy=0.7111, Precision=0.4555, Recall=0.4519
┌──────────────────────┬─────────────────────┬───────────────────┐
│                      │ Predicted stayed    │ Predicted churned │
├──────────────────────┼─────────────────────┼───────────────────┤
│ Actual stayed (1035) │ 833  correct        │ 202  false alarm  │
├──────────────────────┼─────────────────────┼───────────────────┤
│ Actual churned (374) │ 205  MISSED CHURNER │ 169  correct      │
└──────────────────────┴─────────────────────┴───────────────────┘

K=10: Accuracy=0.7928, Precision=0.7071, Recall=0.3743
┌──────────────────────┬─────────────────────┬───────────────────┐
│                      │ Predicted stayed    │ Predicted churned │
├──────────────────────┼─────────────────────┼───────────────────┤
│ Actual stayed (1035) │ 977  correct        │ 58  false alarm   │
├──────────────────────┼─────────────────────┼───────────────────┤
│ Actual churned (374) │ 234  MISSED CHURNER │

# Conclusion


- Larger k means fewer false alarms but more missed churners. The model gets more cautious: it flags fewer people, so precision goes up and recall goes down.
- Accuracy goes up, but only because of the majority class. At k=10, accuracy reaches 0.793, yet the model catches only 140 of 374 churners (37%). The "always predicts stayed" baseline scores 0.735 (1035 / 1409), so even the best accuracy here is only about 6 points above guessing.
- k=1 has the best recall, 0.452. It's also the noisiest, with 202 false alarms and the lowest accuracy.
- No k gets recall above about 0.45. Changing k just moves along the same trade-off, so it doesn't solve the problem.

Which k is best depends on the cost of each mistake:
- If a missed churner costs a lot (a lost customer) and a false alarm is cheap (an unneeded discount offer), lower k is better.
- If retention offers are expensive, a higher k with fewer, more reliable flags is better.

Since k isn't the problem, the likely cause is the unscaled features. To fix this we can use StandardScaler.